# Supervised Learning Overview

**Topic:** Machine Learning Foundations

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import Dropdown, Output, HBox, VBox
from IPython.display import display, clear_output
from sklearn.datasets import make_classification, make_regression
np.random.seed(42)
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** the difference between regression and classification and identify which applies to a given problem
- **Explain** the supervised learning paradigm: what labeled data is and how an algorithm learns from it
- **Interpret** an algorithm positioning chart showing every supervised algorithm on the interpretability-complexity spectrum

> **Tip:** Use the algorithm selector widget below to explore which algorithms suit different problem types. You will build each of these from scratch in the notebooks that follow.

---
## How we got here

You have covered a lot of ground to reach this point. Here is where each piece fits:

- **[tools/](../tools/)** — set up your coding environment and package manager
- **[python/](../python/)** — Python fundamentals: data types, loops, functions, classes
- **[statistics/](../statistics/)** — descriptive statistics, probability, distributions, hypothesis testing
- **[math/](../math/)** — linear algebra, calculus, and the mathematical foundations of learning
- **[eda/](../eda/)** — loading data, exploring structure, spotting patterns visually
- **[preprocessing/](../preprocessing/)** — cleaning, scaling, encoding, handling missing values
- **[ml_concepts/](../ml_concepts/)** — train/test split, bias-variance tradeoff, overfitting, generalization, hyperparameters
- **[feature_engineering/](../feature_engineering/)** — transforming raw inputs into useful signals for models

Every concept from those folders will be applied here. Supervised learning is where all of those pieces come together in service of one goal: predicting an output from labeled inputs.

---
## Why this matters for data science

Supervised learning is the most widely used category of machine learning in practice. Fraud detection, disease diagnosis, housing price estimation, customer churn modeling, credit scoring: the vast majority of real business ML problems involve a labeled dataset and a target variable you want to predict.

Understanding the full landscape of algorithms helps you make better choices. Every algorithm involves trade-offs between interpretability and accuracy, between training speed and predictive power, and between simplicity and flexibility. This folder builds your vocabulary for those trade-offs.

By the end you will not just know how to fit a model. You will know which one to reach for, why, and what to watch out for.

In [2]:
out = Output()

ALGO_INFO = {
    "Linear Regression": {
        "problem": ["Regression"],
        "interp": "High",
        "sizes": ["Small", "Medium", "Large"],
        "speed": "Fast",
        "rationale": "Maximum interpretability and fast training — best when the linear assumption holds",
    },
    "Logistic Regression": {
        "problem": ["Classification"],
        "interp": "High",
        "sizes": ["Small", "Medium", "Large"],
        "speed": "Fast",
        "rationale": "Fully interpretable probability outputs — the standard baseline for classification",
    },
    "Decision Tree": {
        "problem": ["Regression", "Classification", "Either"],
        "interp": "High",
        "sizes": ["Small", "Medium"],
        "speed": "Fast",
        "rationale": "Human-readable split rules — ideal when stakeholders need to trace every decision",
    },
    "K-Nearest Neighbors": {
        "problem": ["Regression", "Classification", "Either"],
        "interp": "Medium",
        "sizes": ["Small", "Medium"],
        "speed": "Fast",
        "rationale": "No training step, intuitive logic — works well for small, low-dimensional datasets",
    },
    "Random Forest": {
        "problem": ["Regression", "Classification", "Either"],
        "interp": "Low",
        "sizes": ["Medium", "Large"],
        "speed": "No preference",
        "rationale": "Robust and accurate on tabular data — a reliable default when interpretability is secondary",
    },
    "Gradient Boosting": {
        "problem": ["Regression", "Classification", "Either"],
        "interp": "Low",
        "sizes": ["Medium", "Large"],
        "speed": "No preference",
        "rationale": "Top performer on structured data — requires careful tuning, not for regulated settings",
    },
    "Neural Network": {
        "problem": ["Regression", "Classification", "Either"],
        "interp": "Low",
        "sizes": ["Large"],
        "speed": "No preference",
        "rationale": "Captures any pattern given enough data — use only when simpler models fall short",
    },
}

INTERP_RANK = {"High": 3, "Medium": 2, "Low": 1}

prob_dd = Dropdown(
    options=["Regression", "Classification", "Either"], value="Either",
    description="Problem type:", style={"description_width": "130px"},
    layout=widgets.Layout(width="360px"))
interp_dd = Dropdown(
    options=["High", "Medium", "Low"], value="Medium",
    description="Interpretability:", style={"description_width": "130px"},
    layout=widgets.Layout(width="360px"))
size_dd = Dropdown(
    options=["Small (<1k)", "Medium", "Large (>100k)"], value="Medium",
    description="Dataset size:", style={"description_width": "130px"},
    layout=widgets.Layout(width="360px"))
speed_dd = Dropdown(
    options=["Fast required", "No preference"], value="No preference",
    description="Training speed:", style={"description_width": "130px"},
    layout=widgets.Layout(width="360px"))

def score_algo(info, problem, interp_need, size, speed):
    s = 0
    if problem in info["problem"] or "Either" in info["problem"]:
        s += 1
    if INTERP_RANK[info["interp"]] >= INTERP_RANK[interp_need]:
        s += 1
    size_key = size.split(" ")[0]
    if size_key in info["sizes"]:
        s += 1
    if speed == "No preference" or info["speed"] == "Fast":
        s += 1
    return s

def render(change=None):
    problem     = prob_dd.value
    interp_need = interp_dd.value
    size        = size_dd.value
    speed       = speed_dd.value

    scores      = {name: score_algo(info, problem, interp_need, size, speed)
                   for name, info in ALGO_INFO.items()}
    sorted_algos = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    names  = [a[0] for a in sorted_algos]
    vals   = [a[1] for a in sorted_algos]
    best_score = max(vals)
    colors = [
        PALETTE["primary"]   if v == best_score else
        PALETTE["secondary"] if v >= 3 else
        PALETTE["muted"]
        for v in vals
    ]
    best_name    = names[0]
    rationale    = ALGO_INFO[best_name]["rationale"]

    traces = [go.Bar(x=names, y=vals, marker_color=colors,
                     text=vals, textposition="outside")]
    layout = base_layout(
        title=f"Best match: {best_name} — {rationale}",
        xaxis_title="Algorithm",
        yaxis_title="Fit score (out of 4)",
    )
    layout.update(yaxis=dict(range=[0, 5.2]), showlegend=False)
    with out:
        clear_output(wait=True)
        fig = go.Figure(data=traces, layout=layout)
        display(go.FigureWidget(fig))

for dd in [prob_dd, interp_dd, size_dd, speed_dd]:
    dd.observe(render, names="value")

display(VBox([HBox([prob_dd, interp_dd]), HBox([size_dd, speed_dd]), out]))
render()

---
## What's happening?

**Supervised learning** means every training example carries a label: the correct answer. The algorithm studies thousands of (features, label) pairs, searching for the pattern that lets it predict the label from the features alone. Once trained, it applies that pattern to new, unseen examples.

### Two kinds of problems

The type of the label determines which family of algorithms applies:

| Problem type | Label | Goal | Examples |
|---|---|---|---|
| **Regression** | Continuous number | Predict a quantity | House price, temperature, sales revenue |
| **Classification** | Category | Predict which class | Spam or not spam, disease or healthy, digit 0-9 |

Some problems can be framed either way. Predicting whether a patient will be readmitted to hospital is classification. Predicting how many days until readmission is regression.

### The four stages of every supervised model

1. **Load and split** — separate data into train and test sets before touching the model
2. **Fit** — show the algorithm the training examples; it adjusts internal parameters to minimize prediction error
3. **Predict** — apply the trained model to the held-out test set
4. **Evaluate** — measure how well predictions match the true labels using a metric suited to your problem type

---
## Real-world example: Where every algorithm sits

The chart below positions the algorithms you will study on two dimensions that matter for every real project: **interpretability** (can you explain why the model made a prediction?) and **model complexity** (can it capture intricate nonlinear patterns?).

- **Notice:** Simple algorithms in the top-left are often the best first try — they train fast, are easy to debug, and establish a strong baseline to beat
- **Notice:** Complex algorithms in the bottom-right can capture subtler patterns but are harder to explain and more prone to overfitting without careful tuning
- **Notice:** The best algorithm depends on your constraints: data size, interpretability requirements, available compute, and tuning time

> **Discussion question:** If you are building a loan approval model at a regulated bank that must explain every decision to applicants, which region of this chart would you focus on?

### Algorithms in this folder at a glance

| Algorithm | Type | One-sentence summary |
|---|---|---|
| Linear Regression | Regression | Fits the best straight line through labeled data |
| Multiple Linear Regression | Regression | Extends the line to many input features |
| Polynomial Regression | Regression | Fits curves by adding powers of input features |
| Ridge / Lasso | Regression | Adds a penalty to prevent overfitting |
| Logistic Regression | Classification | Turns a linear score into a probability |
| K-Nearest Neighbors | Both | Predicts by looking at the K closest training examples |
| Decision Trees | Both | Asks a series of yes/no questions about features |
| Random Forests | Both | Averages many decision trees built on random subsets |
| Gradient Boosting | Both | Builds trees one at a time, each fixing the last one's errors |
| XGBoost | Both | Gradient boosting with regularization and speed optimizations |
| Support Vector Machines | Both | Finds the widest possible boundary between classes |
| Naive Bayes | Classification | Updates class probabilities using Bayes theorem |

In [3]:
np.random.seed(42)

algorithms = [
    ("Linear Regression",       0.95, 0.10, "Regression",     PALETTE["primary"]),
    ("Ridge / Lasso",           0.92, 0.14, "Regression",     PALETTE["primary"]),
    ("Polynomial Regression",   0.74, 0.38, "Regression",     PALETTE["primary"]),
    ("Logistic Regression",     0.90, 0.15, "Classification", PALETTE["secondary"]),
    ("Naive Bayes",             0.70, 0.20, "Classification", PALETTE["secondary"]),
    ("K-Nearest Neighbors",     0.60, 0.43, "Both",           PALETTE["accent"]),
    ("Decision Tree",           0.85, 0.52, "Both",           PALETTE["accent"]),
    ("SVM",                     0.45, 0.72, "Both",           PALETTE["muted"]),
    ("Random Forest",           0.55, 0.73, "Both",           PALETTE["accent"]),
    ("Gradient Boosting",       0.35, 0.85, "Both",           PALETTE["accent"]),
    ("XGBoost",                 0.28, 0.93, "Both",           PALETTE["accent"]),
]

seen = set()
traces = []
for name, interp, cmplx, kind, color in algorithms:
    traces.append(go.Scatter(
        x=[cmplx], y=[interp],
        mode="markers+text",
        marker=dict(size=18, color=color, opacity=0.85,
                    line=dict(width=1.5, color="#ffffff")),
        text=[name], textposition="top center",
        textfont=dict(size=10, family=FONT["family"]),
        name=kind, legendgroup=kind,
        showlegend=(kind not in seen),
    ))
    seen.add(kind)

layout = base_layout(
    title="Supervised Algorithms: Interpretability vs Complexity",
    xaxis_title="Model Complexity →",
    yaxis_title="Interpretability →",
)
layout.update(
    xaxis=dict(range=[-0.05, 1.05], showticklabels=False),
    yaxis=dict(range=[-0.05, 1.20], showticklabels=False),
    height=540,
)
fig = go.Figure(data=traces, layout=layout)
display(go.FigureWidget(fig))

FigureWidget({
    'data': [{'legendgroup': 'Regression',
              'marker': {'color': '#4C6EF5', 'line': {'color': '#ffffff', 'width': 1.5}, 'opacity': 0.85, 'size': 18},
              'mode': 'markers+text',
              'name': 'Regression',
              'showlegend': True,
              'text': [Linear Regression],
              'textfont': {'family': 'Inter, Arial, sans-serif', 'size': 10},
              'textposition': 'top center',
              'type': 'scatter',
              'uid': '3aaa44a7-3bc9-42df-b934-c23847aa6784',
              'x': [0.1],
              'y': [0.95]},
             {'legendgroup': 'Regression',
              'marker': {'color': '#4C6EF5', 'line': {'color': '#ffffff', 'width': 1.5}, 'opacity': 0.85, 'size': 18},
              'mode': 'markers+text',
              'name': 'Regression',
              'showlegend': False,
              'text': [Ridge / Lasso],
              'textfont': {'family': 'Inter, Arial, sans-serif', 'size': 10},
              

> **Supervised learning turns labeled examples into prediction rules — the algorithm's job is to find the pattern in your training data that generalizes to data it has never seen.**

---
*Next up: 01 — Linear Regression, the simplest and most interpretable supervised algorithm*